# Experiment 2b — Stage-2 fine-tune: Mistral-7B-v0.1 classifier

Fine-tunes **Mistral-7B-v0.1** — the *same base weights as the released
DataSentinel-7B KAD adapter* — into an in-distribution constrained-decoding
**benign/injection classifier** (`p_safe`). The headline comparison:

> On identical Mistral-7B-v0.1 weights, the released KAD fine-tune collapses to
> **62–65% benign FPR** (measured) while our classifier fine-tune achieves **[X]**.

Base held constant; only training data + objective differ. See
`results/analysis/STAGE2_BASE_MODEL_DESIGN_NOTE.md` (D1/D2).

**Training set:** `data/train_proposal/train_stage2.jsonl` — the
proposal train augmented with ~5k long (1k–5k char) eval-disjoint benign to
fix the **train/eval benign-length confound** that reproduces DataSentinel's
FPR failure (bd memory `stage2-train-benign-length-confound`).

**Recipe:** QLoRA 4-bit NF4, `configs/models/mistral-7b-v0.1.yaml` +
`configs/training_7b.yaml` (per-device batch 1 × grad-accum 32 = effective
batch 32; 3 epochs; lr 2e-4 cosine; seq 2048). Reuses `scripts/train_stage1.py`
unchanged — 7B is a config change, not new code.

**Outputs** (`results/stage2/mistral-7b-v0.1/`): `adapter/`,
`val_logits.jsonl` (5,724), `cal_logits.jsonl` (5,722), `train_summary.json`.

**VRAM:** 7B QLoRA at 4-bit needs a **24 GB-class GPU** (A100/L4). Cloud-GPU
step — run on Colab; verify artifacts before sign-off.

**Payload hygiene (CLAUDE.md):** no dataset text printed — counts / lengths /
`p_safe` ranges / paths only. Clear outputs before saving.

In [ ]:
# ── Environment detection + installs ────────────────────────────────────
# Auto-detects Google Colab and installs the QLoRA training stack there.
# On a local machine this cell is a no-op (uses the project environment).
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    %pip install -q -U bitsandbytes accelerate peft transformers trl datasets
    # Colab preinstalls torchao 0.10, below peft's minimum (>=0.16); peft raises
    # on a merely-present old torchao even though we never use it (quantization
    # is bitsandbytes). Removing is safer than upgrading torch's CUDA build.
    %pip uninstall -q -y torchao
    print("Colab detected — training dependencies installed.")
else:
    print(f"Local run ({sys.platform}) — using the project environment as-is.")


In [ ]:
# ── Config ─────────────────────────────────────────────────────────
# BASELINE_SPEC.md §Environment ; results/analysis/STAGE2_BASE_MODEL_DESIGN_NOTE.md

RUN_MODE = "smoke"   # "smoke" | "full"   <- set "full" for the GPU Colab run
SEED     = 3131

import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # Project synced to Drive; scripts/ configs/ data/ live inside it.
    REPO_ROOT = Path("/content/drive/MyDrive/Thesis")
    # Gated bases (llama3.2, mistral) need an HF token — Colab Secrets (key icon).
    if "HF_TOKEN" not in os.environ:
        try:
            from google.colab import userdata
            os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
        except Exception:
            pass
else:
    _here = Path(globals().get("__vsc_ipynb_file__", Path.cwd() / "_")).resolve()
    REPO_ROOT = next((p for p in _here.parents if (p / "BASELINE_SPEC.md").exists()), Path.cwd())

# make repo importable (src/ , scripts/)
for _p in (str(REPO_ROOT), str(REPO_ROOT / "src")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

BASE = "mistral-7b-v0.1"
CONFIG_PATH  = REPO_ROOT / "configs" / "models" / f"{BASE}.yaml"
TRAINING_CFG = REPO_ROOT / "configs" / "training_7b.yaml"
TRAIN_FILE   = REPO_ROOT / "data" / "train_proposal" / "train_stage2.jsonl"
VAL_FILE     = REPO_ROOT / "data" / "train_proposal" / "val.jsonl"
MANIFEST     = REPO_ROOT / "data" / "train_proposal" / "stage2_aug_manifest.json"
OUTPUT_DIR   = REPO_ROOT / "results" / "stage2" / BASE
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert CONFIG_PATH.exists(), f"missing {CONFIG_PATH}"
assert TRAINING_CFG.exists(), f"missing {TRAINING_CFG}"
assert TRAIN_FILE.exists(), (
    f"missing {TRAIN_FILE} — build it first: python scripts/build_stage2_trainset.py")

print(f"IN_COLAB={IN_COLAB}  RUN_MODE={RUN_MODE}  SEED={SEED}")
print(f"REPO_ROOT={REPO_ROOT}")


## Device detection

QLoRA 4-bit needs a CUDA GPU (bitsandbytes is CUDA-only). On MPS/CPU the
notebook falls back to a 1-step **dry-run** that validates the pipeline without
real training — the real run is the `cloud-gpu` Colab pass.

In [ ]:
import torch

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

CUDA_AVAILABLE = (DEVICE == "cuda")
# Real training only on CUDA + RUN_MODE="full"; otherwise a 1-step dry-run.
DRY_RUN = not (CUDA_AVAILABLE and RUN_MODE == "full")
print(f"Detected device: {DEVICE}   CUDA={CUDA_AVAILABLE}   DRY_RUN={DRY_RUN}")

if DRY_RUN:
    print(
        "\n" + "=" * 70 + "\n"
        "Real QLoRA training needs a CUDA GPU + RUN_MODE='full'.\n"
        f"This machine reports device={DEVICE}, RUN_MODE={RUN_MODE}.\n"
        "\nProceeding in DRY-RUN mode: 1 optimizer step on a tiny subset to\n"
        "validate the pipeline (no logit dumps, no selection). For the real run:\n"
        "  1. Sync the project to Google Drive at MyDrive/Thesis.\n"
        "  2. Open in Colab with a GPU runtime, set RUN_MODE='full', Run all.\n"
        + "=" * 70
    )


In [ ]:
import subprocess

def run_train(config_path, out_dir, train_file=None, training_cfg=None):
    """Shell out to the canonical, tested trainer (scripts/train_stage1.py).
    Returns the process return code; streams logs live."""
    cmd = [sys.executable, str(REPO_ROOT / "scripts" / "train_stage1.py"),
           "--config", str(config_path), "--output-dir", str(out_dir)]
    if train_file is not None:
        cmd += ["--train-file", str(train_file)]
    if training_cfg is not None:
        cmd += ["--training", str(training_cfg)]
    if DRY_RUN:
        cmd += ["--dry-run"]
    print("$ " + " ".join(cmd), flush=True)
    proc = subprocess.run(cmd, cwd=str(REPO_ROOT))
    return proc.returncode


In [ ]:
import json, math

def sanity_logits(out_dir, n_val=5724, n_cal=5722):
    """Verify val/cal logit dumps match the split row counts and p_safe range.
    Payload-safe: reads only {logp_benign, logp_injection, p_safe}, never text."""
    report = {}
    for stem, n_exp in (("val", n_val), ("cal", n_cal)):
        fp = Path(out_dir) / f"{stem}_logits.jsonl"
        if not fp.exists():
            report[stem] = {"exists": False}
            continue
        n = 0; bad = 0; pmin = 1.0; pmax = 0.0
        with fp.open() as fh:
            for line in fh:
                if not line.strip():
                    continue
                r = json.loads(line); n += 1
                p = r.get("p_safe")
                if p is None or not (0.0 <= p <= 1.0) or math.isnan(p) \
                   or "logp_benign" not in r or "logp_injection" not in r:
                    bad += 1
                else:
                    pmin = min(pmin, p); pmax = max(pmax, p)
        report[stem] = {"exists": True, "rows": n, "expected": n_exp,
                        "rows_ok": n == n_exp, "bad_rows": bad,
                        "p_safe_min": round(pmin, 4), "p_safe_max": round(pmax, 4)}
    return report


In [ ]:
from evaluation.metrics import detection_rate_at_fpr

_SAFE = {"safe", "benign", 0, "0"}

def _load_val_meta(val_path):
    """label(0/1) + channel per val row, IN ORDER. Never reads `input`."""
    labels, channels = [], []
    with open(val_path) as fh:
        for line in fh:
            if not line.strip():
                continue
            r = json.loads(line)
            labels.append(0 if r.get("label") in _SAFE else 1)
            channels.append(r.get("channel"))
    return labels, channels

def _load_scores(logits_path):
    """injection score = 1 - p_safe, IN ORDER."""
    scores = []
    with open(logits_path) as fh:
        for line in fh:
            if not line.strip():
                continue
            scores.append(1.0 - json.loads(line)["p_safe"])
    return scores

def eval_candidate(val_path, logits_path):
    labels, channels = _load_val_meta(val_path)
    scores = _load_scores(logits_path)
    assert len(labels) == len(scores), f"row mismatch: {len(labels)} vs {len(scores)}"
    dr = detection_rate_at_fpr(labels, scores, fpr_targets=(0.01,))["0.01"]
    thr = dr["threshold"]
    # per-channel injection recall at the global 1%-FPR threshold
    per_ch = {}
    from collections import defaultdict
    hit = defaultdict(int); tot = defaultdict(int)
    for lab, ch, sc in zip(labels, channels, scores):
        if lab == 1:
            tot[ch] += 1
            if sc > thr:
                hit[ch] += 1
    for ch in tot:
        per_ch[ch] = round(hit[ch] / tot[ch], 4) if tot[ch] else None
    return {"dr_at_1pct_fpr": round(dr["dr"], 4),
            "achieved_fpr": round(dr["achieved_fpr"], 4),
            "threshold": round(thr, 4), "per_channel_recall": per_ch}


## Preflight — augmented train + tokenizer boundary

In [ ]:
# ── Preflight: augmented-train stats + tokenizer boundary ───────────────
# (a) confirm the length augmentation is present (aggregate-only, from manifest)
if MANIFEST.exists():
    man = json.loads(MANIFEST.read_text())
    bb, ba = man["benign_before"], man["benign_after"]
    print(f"benign len p90: {bb['len_p90']} -> {ba['len_p90']}   "
          f"added {man['added']['n_total']} long benign   "
          f"eval collisions removed: {man['dedup']['removed_eval_collisions']}")
    print(f"injection untouched: {man['counts']['injection_untouched']}")
else:
    print("(manifest not found — skipping augmentation summary)")

# (b) tokenizer boundary: the label must tokenize identically alone (Stage1Detector
# scoring) and as the head of the completion (TRL training). This is what makes the
# constrained-decoding p_safe well-defined for a base (non-instruct) Mistral.
from transformers import AutoTokenizer
from models.prompt_template import load_model_config, format_training_example

_cfg = load_model_config(str(CONFIG_PATH))
_tok = AutoTokenizer.from_pretrained(_cfg.hf_id)
for _label in _cfg.labels:
    _p, _c = format_training_example(_cfg, "Summarize the following report.", _label)
    _pids = _tok(_p, add_special_tokens=False)["input_ids"]
    _cids = _tok(_c, add_special_tokens=False)["input_ids"]
    _lids = _tok(_label, add_special_tokens=False)["input_ids"]
    _score = _pids + _lids
    ok = (_pids + _cids)[:len(_score)] == _score and _cids[-1] == _tok.eos_token_id
    assert ok, f"tokenizer boundary broken for label={_label!r}"
    print(f"  boundary OK: label={_label:9s} n_prompt={len(_pids)} n_label={len(_lids)}")


## Fine-tune

In [ ]:
# ── Fine-tune ──────────────────────────────────────────────────
# Reuses scripts/train_stage1.py at 7B (config change only). Writes
# results/stage2/mistral-7b-v0.1/ {adapter, val_logits.jsonl, cal_logits.jsonl,
# train_summary.json}. On a full GPU run this is the long step (~30-90 min L4/A100).
rc = run_train(CONFIG_PATH, OUTPUT_DIR, train_file=TRAIN_FILE, training_cfg=TRAINING_CFG)
assert rc == 0, f"training failed (rc={rc})"
print(f"\ntrain_stage1.py return code = {rc}")
if (OUTPUT_DIR / "train_summary.json").exists():
    print(json.dumps(json.loads((OUTPUT_DIR / "train_summary.json").read_text()), indent=2))


## Sanity — logit dumps

In [ ]:
# ── Sanity: logit dumps match split row counts + p_safe range ─────────────
rep = sanity_logits(OUTPUT_DIR)
print(f"{BASE}: {rep}")
if not DRY_RUN:
    for stem in ("val", "cal"):
        assert rep[stem]["exists"] and rep[stem]["rows_ok"], f"{stem} row count wrong"
        assert rep[stem]["bad_rows"] == 0, f"{stem} has out-of-range/NaN p_safe"
else:
    print("\n(DRY-RUN: logit dumps skipped — asserts deferred to the full run.)")


## Gate preview — val benign FPR

In [ ]:
# ── Gate preview: benign FPR + DR@1%FPR on val vs DataSentinel ~62-65% ──────
# Immediate signal on whether the length-augmentation fixed the FPR failure.
# The formal go/no-go gate is the Stage-2 val gate (results/metrics/stage2_val_gate.json).
if DRY_RUN:
    print("DRY-RUN: no val_logits to gate on. Run RUN_MODE='full' on a 24GB GPU first.")
else:
    m = eval_candidate(VAL_FILE, OUTPUT_DIR / "val_logits.jsonl")
    print(f"DR@1%FPR (val)   : {m['dr_at_1pct_fpr']:.4f}")
    print(f"achieved FPR     : {m['achieved_fpr']:.4f}  (target 1%)")
    print(f"per-channel recall @1%FPR: {m['per_channel_recall']}")
    print("\nDataSentinel-7B benign FPR baseline: ~0.62-0.65 (measured).")
    print("A working stage 2 holds ~1% FPR at high injection recall on the")
    print("uncontaminated direct / tool-output channels.")


## Summary

In [ ]:
# ── Final summary ─────────────────────────────────────────────
print("=" * 60)
print("Stage-2 Mistral-7B-v0.1 fine-tune — run complete")
print("=" * 60)
print(f"RUN_MODE : {RUN_MODE}    DRY_RUN : {DRY_RUN}    device : {DEVICE}")
print(f"base     : mistralai/Mistral-7B-v0.1 (same weights as DataSentinel-7B)")
print(f"train    : {TRAIN_FILE.name}")
print(f"outputs  : {OUTPUT_DIR}")
for f in ("adapter", "val_logits.jsonl", "cal_logits.jsonl", "train_summary.json"):
    print(f"  {f:22s} {(OUTPUT_DIR / f).exists()}")


## Colab instructions (full run)

1. **Sync to Drive.** Project at `MyDrive/Thesis` with `scripts/`, `configs/`,
   `src/`, and `data/train_proposal/train_stage2.jsonl` (+ `val.jsonl`,
   `cal.jsonl`). Outputs land in `MyDrive/Thesis/results/stage2/mistral-7b-v0.1/`.
2. **HF token.** Add `HF_TOKEN` in Colab Secrets — `mistralai/Mistral-7B-v0.1`
   is a gated repo; accept its license on Hugging Face once.
3. **24 GB GPU.** Runtime → GPU → **A100 or L4** (7B QLoRA won't fit a T4's
   16 GB comfortably). Set `RUN_MODE="full"`.
4. **Run all.** Preflight → fine-tune (~30–90 min) → sanity → gate preview.
5. **Verify before sign-off.** `results/stage2/mistral-7b-v0.1/` has
   `adapter/`, `val_logits.jsonl` (5,724), `cal_logits.jsonl` (5,722),
   `train_summary.json`; `p_safe ∈ [0,1]`, no NaN. The formal val gate is separate.